# 04 — VRAM Batch-Size Finder (real probe, multi-GPU aware)

Finds the **largest per-GPU batch size** that trains without running out of memory,
by launching real 1-epoch probes in isolated subprocesses (`scripts/batch_finder.py`),
then refining around the limit. The result is **scaled to all GPUs** of the profile
(per-GPU batch × number of GPUs) and cached to `runs/optimal_batch_<size>_<imgsz>.json`.

**You don't configure anything here** — model size, image size and classes all come from
`scripts/config.py`. Edit that one file, then just run this notebook.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/shared/s0598584/scripts')
import config as C
import train_pipeline as P

# Build a run spec from the central config (model size / imgsz / dataset).
spec = P.make_spec('batchfind', data=C.dataset_yaml(), imgsz=C.IMGSZ, model_size=C.MODEL_SIZE)
device, nproc = P.resolve_devices()
print('model      :', f"{C.BASE_MODEL}{C.MODEL_SIZE}", '| imgsz:', C.IMGSZ)
print('GPUs       :', device, f'({nproc})')

# Real per-GPU probe + refine, scaled to all GPUs. force=True re-measures.
total_batch, info = P.find_optimal_batch(spec, nproc, force=True)
print('\nper-GPU batch :', info['per_gpu_batch'], f"(peak {info['peak_vram_gb']} / {info['total_vram_gb']} GB)")
print('TOTAL batch   :', total_batch, f"across {nproc} GPU")
print('written       : runs/optimal_batch_%s_%d.json' % (C.MODEL_SIZE, C.IMGSZ))

## ✅ Phase 4 done

`runs/optimal_batch_<size>_<imgsz>.json` now holds a **measured** per-GPU batch plus the
total scaled across all GPUs. Continue with `05_train.ipynb`.